# LightGBM Stage 2 — Training Notebook

**What this notebook does (in order):**
1. Loads all wet-station records (1961–2023)
2. Creates a reproducible 5-fold random station split → saves `fold_assignment.parquet` to S3
3. For each fold: computes leakage-free geo-features → uploads to S3 → trains 11 quantile models → uploads to S3
4. Trains final models on all data → uploads to S3

**Resume-safe:** if a fold's features/models are already on S3, it downloads them instead of recomputing.

**S3 layout:**
```
s3://thesis-data-ismaktam/lgbm/
  fold_assignment.parquet          ← station_id → fold_id mapping
  kfold/
    fold0_features.parquet
    fold0_models.joblib
    ...                            (fold1…fold4)
    cv_results.json
  final/
    features.parquet
    models.joblib
```

**Local layout:** `outputs/lgbm/` mirrors the S3 structure.

## 0. Imports

In [2]:
import sys, os, gc, json, warnings
import subprocess, tempfile, shutil, textwrap
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import properscoring as ps
import pyarrow as pa
import pyarrow.parquet as pq
import joblib
import boto3
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

ROOT = Path('../..')
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'Working directory: {Path.cwd()}')
print(f'LightGBM: {lgb.__version__}')
print(f'NumPy:    {np.__version__}')


Working directory: /root/precip_interpolation_thesis
LightGBM: 4.5.0
NumPy:    2.4.4


## 1. Constants

> **Before running on GPU:** set `DEVICE = 'cuda'`  
> **Before running on CPU (local):** set `DEVICE = 'cpu'`

In [3]:
# ── Reproducibility (DO NOT CHANGE) ──────────────────────────────────────────
RANDOM_SEED = 42

# ── Cross-validation ─────────────────────────────────────────────────────────
K_FOLDS   = 5
QUANTILES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# ── Geo-features ─────────────────────────────────────────────────────────────
K_GEO         = 15
SVD_QUANTILES = np.arange(0.0, 1.05, 0.05)        # 21 values → svd_00 … svd_20
SVD_COLS      = [f'svd_{i:02d}' for i in range(len(SVD_QUANTILES))]

# ── LightGBM ─────────────────────────────────────────────────────────────────
NUM_BOOST_ROUND   = 1000
EARLY_STOPPING    = 50

DEVICE = 'cuda'    # 'cuda' on Vast.ai GPU  |  'cpu' on local

LGB_BASE = dict(
    objective         = 'quantile',
    metric            = 'quantile',
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 50,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    device_type       = DEVICE,
    num_threads       = -1,
    seed              = RANDOM_SEED,
    verbose           = -1,
)

# ── Batch size for feature writes (keeps RAM manageable) ──────────────────────
BATCH_SIZE = 2000          # dates per parallel batch
N_JOBS     = os.cpu_count()

# ── Multi-GPU dispatch — 11 quantiles split across visible GPUs ───────────────
# Same architecture as lgbm_hpo.ipynb: each GPU subprocess trains its assigned
# quantiles sequentially (no within-GPU concurrent threads, since 1000-round
# trees are heavier than HPO's 400 rounds and we don't want to risk OOM).
def _detect_gpus():
    try:
        out = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=name,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True,
        )
        rows = [r.strip().split(', ') for r in out.stdout.strip().split('\n') if r.strip()]
        if rows:
            return len(rows), float(rows[0][1]) / 1024.0
    except Exception:
        pass
    return 0, 0.0

_n_detected, GPU_VRAM_GB = _detect_gpus()
N_GPUS = _n_detected if DEVICE == 'cuda' and _n_detected > 0 else 1
N_PARALLEL_QUANTILES = min(N_GPUS, len(QUANTILES))

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR   = Path('outputs/lgbm')
KFOLD_DIR = OUT_DIR / 'kfold'
FINAL_DIR = OUT_DIR / 'final'
for d in [OUT_DIR, KFOLD_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET = 'thesis-data-ismaktam'
S3_ROOT   = 'lgbm'    # all outputs under s3://thesis-data-ismaktam/lgbm/

# Set True to ignore all local/S3 caches and redo everything from scratch
FORCE_RECOMPUTE = False

print(f'RANDOM_SEED={RANDOM_SEED}  K_FOLDS={K_FOLDS}  DEVICE={DEVICE}')
print(f'NUM_BOOST_ROUND={NUM_BOOST_ROUND}  EARLY_STOPPING={EARLY_STOPPING}')
print(f'N_JOBS={N_JOBS}  BATCH_SIZE={BATCH_SIZE}')
if DEVICE == 'cuda':
    print(f'GPUs detected: {_n_detected}  (VRAM: {GPU_VRAM_GB:.1f} GB / GPU)')
    print(f'Quantile dispatch: {len(QUANTILES)} quantiles across {N_PARALLEL_QUANTILES} GPU(s) '
          f'(~{(len(QUANTILES) + N_PARALLEL_QUANTILES - 1) // N_PARALLEL_QUANTILES} '
          f'sequential trainings per GPU)')
else:
    print(f'CPU mode — single in-process loop')
print(f'Output dir: {OUT_DIR}')
print(f'S3: s3://{S3_BUCKET}/{S3_ROOT}/')


RANDOM_SEED=42  K_FOLDS=5  DEVICE=cuda
NUM_BOOST_ROUND=1000  EARLY_STOPPING=50
N_JOBS=48  BATCH_SIZE=2000
GPUs detected: 3  (VRAM: 15.9 GB / GPU)
Quantile dispatch: 11 quantiles across 3 GPU(s) (~4 sequential trainings per GPU)
Output dir: outputs/lgbm
S3: s3://thesis-data-ismaktam/lgbm/


In [4]:
# ── Best hyperparameters from Tyralis-style grid search ─────────────────────
# Source: results/lgbm/hparam/best_params.json (240 combos, CRPS_val=0.5195)
LGB_BASE.update({
    'max_depth':         10,
    'num_leaves':        200,
    'learning_rate':     0.05,
    'min_child_samples': 100,
})
print('LGB_BASE (HPO best):', {k: LGB_BASE[k] for k in
      ('max_depth', 'num_leaves', 'learning_rate', 'min_child_samples')})


LGB_BASE (HPO best): {'max_depth': 10, 'num_leaves': 200, 'learning_rate': 0.05, 'min_child_samples': 100}


In [5]:
# ── GPU worker module ─────────────────────────────────────────────────────────
# Trains a subset of quantiles on a single GPU. Written to disk so it can be
# imported by ProcessPoolExecutor's spawn workers (which start fresh Python
# processes with no inherited CUDA context).
_WORKER_DIR = OUT_DIR / '_gpu_worker'
_WORKER_DIR.mkdir(parents=True, exist_ok=True)
_WORKER_PY = _WORKER_DIR / '_lgbm_train_worker.py'
_WORKER_PY.write_text(textwrap.dedent("""\
    import os, gc
    import numpy as np
    import lightgbm as lgb


    def run_gpu_quantile_batch(args):
        \"\"\"Train assigned quantile models on one GPU. Returns serialized models + predictions.\"\"\"
        (gpu_id, quantile_indices, quantiles, data_dir, base_params,
         num_boost_round, early_stopping, has_val) = args

        # Isolate this process to a single GPU so the device id appears as 0
        # and the CUDA context is independent from sibling workers.
        if gpu_id >= 0:
            os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu_id)

        X_tr = np.load(f'{data_dir}/X_tr.npy', mmap_mode='r')
        y_tr = np.load(f'{data_dir}/y_tr.npy', mmap_mode='r')
        X_va = y_va = None
        if has_val:
            X_va = np.load(f'{data_dir}/X_va.npy', mmap_mode='r')
            y_va = np.load(f'{data_dir}/y_va.npy', mmap_mode='r')

        # One Dataset reused for every quantile in this batch — saves bin construction.
        dtrain = lgb.Dataset(np.asarray(X_tr), label=np.asarray(y_tr), free_raw_data=False)
        dval   = (lgb.Dataset(np.asarray(X_va), label=np.asarray(y_va),
                              reference=dtrain, free_raw_data=False)
                  if has_val else None)

        results = []
        for qi in quantile_indices:
            alpha  = quantiles[qi]
            params = {**base_params, 'alpha': alpha}
            if params.get('device_type') == 'cuda':
                params['gpu_device_id'] = 0   # CUDA_VISIBLE_DEVICES already remapped

            if has_val:
                booster = lgb.train(
                    params, dtrain,
                    num_boost_round=num_boost_round,
                    valid_sets=[dval],
                    callbacks=[
                        lgb.early_stopping(early_stopping, verbose=False),
                        lgb.log_evaluation(period=-1),
                    ],
                )
                preds     = np.clip(booster.predict(np.asarray(X_va)), 0, None).astype(np.float32)
                best_iter = int(booster.best_iteration)
            else:
                booster = lgb.train(
                    params, dtrain,
                    num_boost_round=num_boost_round,
                    callbacks=[lgb.log_evaluation(period=-1)],
                )
                preds     = None
                best_iter = int(booster.num_trees())

            # Send model back as text — robust across processes, no GPU handle leak.
            results.append({
                'qi':        qi,
                'alpha':     alpha,
                'best_iter': best_iter,
                'preds':     preds,
                'model_str': booster.model_to_string(),
            })
            tag = f'best_iter={best_iter}' if has_val else f'trees={best_iter}'
            print(f'GPU{gpu_id} α={alpha:.2f} done ({tag})', flush=True)

            del booster
            gc.collect()
        return results
"""))

# Make the worker importable so the spawn'd subprocesses can find it.
if str(_WORKER_DIR) not in sys.path:
    sys.path.insert(0, str(_WORKER_DIR))
from _lgbm_train_worker import run_gpu_quantile_batch  # noqa: E402,F401


def train_quantiles_multigpu(X_tr, y_tr, X_va=None, y_va=None,
                             num_boost_round=NUM_BOOST_ROUND,
                             early_stopping=EARLY_STOPPING,
                             base_params=None,
                             label='train'):
    """Train all QUANTILES quantile models, distributing them across N_GPUS GPUs.

    Returns
    -------
    q_preds      : np.ndarray (n_val, n_quantiles) or None when no val set
    models       : dict {alpha: lgb.Booster}    — reconstructed from model_str
    best_iters   : dict {alpha: int}            — best_iteration (or num_trees if no val)
    """
    base_params = dict(base_params or LGB_BASE)
    has_val     = X_va is not None and y_va is not None
    n_workers   = max(1, min(N_PARALLEL_QUANTILES, len(QUANTILES)))

    tmp_dir = tempfile.mkdtemp(prefix=f'lgbm_{label}_')
    try:
        np.save(f'{tmp_dir}/X_tr.npy', X_tr)
        np.save(f'{tmp_dir}/y_tr.npy', y_tr)
        if has_val:
            np.save(f'{tmp_dir}/X_va.npy', X_va)
            np.save(f'{tmp_dir}/y_va.npy', y_va)

        # Round-robin distribution: GPU i gets quantile indices i, i+N, i+2N, ...
        q_batches = [list(range(i, len(QUANTILES), n_workers)) for i in range(n_workers)]

        worker_args = [
            (gpu_id if DEVICE == 'cuda' else -1,
             q_batches[gpu_id], QUANTILES, tmp_dir, base_params,
             num_boost_round, early_stopping, has_val)
            for gpu_id in range(n_workers)
        ]

        if n_workers == 1:
            # Single GPU (or CPU) — run in-process, no subprocess overhead.
            all_results = run_gpu_quantile_batch(worker_args[0])
        else:
            ctx = mp.get_context('spawn')
            with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as pool:
                futures = {pool.submit(run_gpu_quantile_batch, a): a[0] for a in worker_args}
                all_results = []
                for fut in as_completed(futures):
                    all_results.extend(fut.result())
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    # Reassemble — model strings → Booster, predictions → column-by-quantile-index
    models     = {}
    best_iters = {}
    q_preds    = (np.zeros((len(X_va), len(QUANTILES)), dtype=np.float32)
                  if has_val else None)

    for r in all_results:
        models[r['alpha']]     = lgb.Booster(model_str=r['model_str'])
        best_iters[r['alpha']] = r['best_iter']
        if has_val:
            q_preds[:, r['qi']] = r['preds']

    return q_preds, models, best_iters


## 2. Load Data

In [6]:
from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.transforms import ProjectionTransform, IndicatorTransform
from thesis.transforms.pipeline import TransformPipeline
from thesis.scripts.run_grk_kfold_cv import SOIL_VARS
from thesis.models.grk.features import compute_day_geo_features

cfg      = Config()
registry = DataRegistry.from_config(cfg)

print(f'Date range: {cfg.date_start} → {cfg.date_end}')
print('Loading station data...')

all_raw  = registry.stations.load(cfg.date_start, cfg.date_end)
pipeline = TransformPipeline([
    ProjectionTransform(target_crs=cfg.study_area.target_crs),
    IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm),
])
all_proc = pipeline.fit_transform(all_raw)
df_wet   = all_proc[all_proc['rain_indicator'] == 1].copy()

# One row per station (spatial metadata)
station_meta = (
    all_proc
    .drop_duplicates('station_id')[['station_id', 'x_proj', 'y_proj', 'elevation_m']]
    .reset_index(drop=True)
)

print(f'All records:  {len(all_proc):,}')
print(f'Wet records:  {len(df_wet):,}  ({len(df_wet)/len(all_proc):.1%})')
print(f'Stations:     {len(station_meta):,}')
print(f'Wet stations: {df_wet["station_id"].nunique():,}')

Date range: 1961-01-01 → 2023-12-31
Loading station data...
All records:  45,237,660
Wet records:  17,409,376  (38.5%)
Stations:     1,966
Wet stations: 1,966


## 3. SoilGrids + Feature Columns

In [7]:
soil_rows = {'station_id': station_meta['station_id'].values}
for var, src in registry.soilgrids.items():
    if var in SOIL_VARS:
        soil_rows[var] = src.sample_at_projected(
            station_meta['x_proj'].values,
            station_meta['y_proj'].values,
        )

soil_static    = pd.DataFrame(soil_rows).set_index('station_id')
available_soil = [v for v in SOIL_VARS if v in soil_static.columns]
for v in available_soil:
    soil_static[v] = soil_static[v].fillna(float(soil_static[v].median()))

# Feature column order — this must NEVER change between train and eval
FEATURE_COLS = (
    ['x_proj', 'y_proj', 'elevation_m']   # static spatial
    + ['idw', 'gos']                       # geo interpolants
    + SVD_COLS                             # 21 SVD quantiles of neighbours
    + available_soil                       # SoilGrids static
)
TARGET_COL = 'precip_mm'

print(f'SoilGrids vars: {available_soil}')
print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

SoilGrids vars: ['bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa']
Total features: 32
['x_proj', 'y_proj', 'elevation_m', 'idw', 'gos', 'svd_00', 'svd_01', 'svd_02', 'svd_03', 'svd_04', 'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09', 'svd_10', 'svd_11', 'svd_12', 'svd_13', 'svd_14', 'svd_15', 'svd_16', 'svd_17', 'svd_18', 'svd_19', 'svd_20', 'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa']


## 4. S3 Helpers + Fold Assignment

The fold assignment is saved to S3 as `fold_assignment.parquet`.  
The eval notebook must load this file to reproduce the **exact same** folds.

In [10]:
s3 = boto3.client('s3')


def s3_exists(s3_key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        return True
    except Exception:
        return False


def s3_upload(local_path: Path, s3_key: str) -> None:
    try:
        s3.upload_file(str(local_path), S3_BUCKET, s3_key)
        print(f'  ↑ s3://{S3_BUCKET}/{s3_key}')
    except Exception as e:
        print(f'  S3 upload skipped: {e.__class__.__name__}: {e}')


def s3_download(s3_key: str, local_path: Path) -> bool:
    """Download from S3; returns True on success."""
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception:
        return False


# ── Fold assignment ───────────────────────────────────────────────────────────
FOLD_LOCAL = OUT_DIR / 'fold_assignment.parquet'
FOLD_S3    = f'{S3_ROOT}/fold_assignment.parquet'

if FOLD_LOCAL.exists() and not FORCE_RECOMPUTE:
    df_folds = pd.read_parquet(FOLD_LOCAL)
    print(f'Fold assignment: loaded from local cache')
elif not FORCE_RECOMPUTE and s3_download(FOLD_S3, FOLD_LOCAL):
    df_folds = pd.read_parquet(FOLD_LOCAL)
    print(f'Fold assignment: downloaded from S3')
else:
    # Create reproducible fold assignment
    rng      = np.random.default_rng(RANDOM_SEED)
    shuffled = rng.permutation(station_meta['station_id'].values)
    df_folds = pd.DataFrame({
        'station_id': shuffled,
        'fold':       [int(i % K_FOLDS) for i in range(len(shuffled))],
    })
    df_folds.to_parquet(FOLD_LOCAL, index=False)
    s3_upload(FOLD_LOCAL, FOLD_S3)
    print(f'Fold assignment: created and saved to S3')

# Attach fold to station_meta and df_wet
station_meta = station_meta.merge(df_folds[['station_id', 'fold']], on='station_id', how='left')
df_wet       = df_wet.merge(df_folds[['station_id', 'fold']], on='station_id', how='left')

print('Stations per fold:')
print(station_meta['fold'].value_counts().sort_index().to_string())
assert df_wet['fold'].isna().sum() == 0, 'Some wet records have no fold assignment!'

Fold assignment: loaded from local cache
Stations per fold:
fold
0    394
1    393
2    393
3    393
4    393


## 5. Feature Computation Helpers

In [11]:
def _day_features(date, grp, train_sids_set):
    """Compute geo-features for one day using only train_sids as neighbours."""
    grp        = grp.reset_index(drop=True)
    xy         = grp[['x_proj', 'y_proj']].values
    z          = grp['precip_mm'].values
    sids       = grp['station_id'].values
    train_mask = np.array([s in train_sids_set for s in sids])
    return compute_day_geo_features(date, xy, z, sids, train_mask, K_GEO, SVD_QUANTILES)


def write_features(groups, train_sids_set, out_path: Path, label: str) -> int:
    """
    Compute geo-features in parallel batches and stream to a Parquet file.
    Returns total number of rows written.
    """
    n_batches = (len(groups) + BATCH_SIZE - 1) // BATCH_SIZE
    writer    = None
    total     = 0

    for b in range(n_batches):
        batch   = groups[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=0)(
            delayed(_day_features)(date, grp, train_sids_set)
            for date, grp in batch
        )
        records = [r for day_recs in results for r in day_recs]
        del results; gc.collect()

        if not records:
            continue

        df_b  = pd.DataFrame(records)
        del records; gc.collect()
        table = pa.Table.from_pandas(df_b, preserve_index=False)
        del df_b; gc.collect()

        if writer is None:
            writer = pq.ParquetWriter(str(out_path), table.schema)
        writer.write_table(table)
        total += len(table)
        del table; gc.collect()

        print(f'  [{label}] batch {b+1}/{n_batches}  ({total:,} rows)', end='\r')

    if writer:
        writer.close()
    print(f'  [{label}] done — {total:,} rows → {out_path.name}' + ' ' * 20)
    return total


# Pre-compute the list of (date, group) once — reused for every fold
print('Pre-grouping wet records by date...')
groups = list(df_wet.groupby('date'))
print(f'Unique dates: {len(groups):,}')

Pre-grouping wet records by date...
Unique dates: 22,650


## 6. 5-Fold CV: Features + Training

For each fold:
1. Compute leakage-free geo-features (test stations excluded from neighbour pool)
2. Train 11 quantile LightGBM models with early stopping
3. Compute CRPS / MAE on the held-out test fold
4. Save features + models to S3

In [12]:
CV_JSON_LOCAL = KFOLD_DIR / 'cv_results.json'
CV_JSON_S3    = f'{S3_ROOT}/kfold/cv_results.json'

# Resume from checkpoint
if CV_JSON_LOCAL.exists() and not FORCE_RECOMPUTE:
    with open(CV_JSON_LOCAL) as f:
        cv_results = json.load(f)
    done_folds = {r['fold'] for r in cv_results}
    print(f'Resuming CV — folds already done: {sorted(done_folds)}')
else:
    cv_results = []
    done_folds = set()

for fold_id in range(K_FOLDS):
    if fold_id in done_folds:
        print(f'Fold {fold_id}: checkpoint found, skipping')
        continue

    print(f'\n{"="*60}')
    print(f'FOLD {fold_id} / {K_FOLDS - 1}')
    print(f'{"="*60}')

    feat_local = KFOLD_DIR / f'fold{fold_id}_features.parquet'
    feat_s3    = f'{S3_ROOT}/kfold/fold{fold_id}_features.parquet'
    mdl_local  = KFOLD_DIR / f'fold{fold_id}_models.joblib'
    mdl_s3     = f'{S3_ROOT}/kfold/fold{fold_id}_models.joblib'

    # ── 1. Geo-features ───────────────────────────────────────────────────────
    if feat_local.exists() and not FORCE_RECOMPUTE:
        print(f'  Features: local cache')
    elif not FORCE_RECOMPUTE and s3_download(feat_s3, feat_local):
        print(f'  Features: downloaded from S3')
    else:
        print(f'  Features: computing (n_jobs={N_JOBS})...')
        train_sids = set(
            station_meta.loc[station_meta['fold'] != fold_id, 'station_id']
        )
        write_features(groups, train_sids, feat_local, f'fold{fold_id}')
        s3_upload(feat_local, feat_s3)

    # ── 2. Build train / test arrays ──────────────────────────────────────────
    df_geo  = pd.read_parquet(feat_local)
    df_fold = (
        df_wet[['station_id', 'date', 'precip_mm', 'fold',
                'x_proj', 'y_proj', 'elevation_m']]
        .merge(df_geo, on=['station_id', 'date'], how='inner')
        .merge(soil_static[available_soil].reset_index(), on='station_id', how='left')
    )
    # Fill any remaining NaNs in soil columns with column median
    for v in available_soil:
        med = df_fold[v].median()
        df_fold[v] = df_fold[v].fillna(med)

    train_mask = df_fold['fold'] != fold_id
    test_mask  = df_fold['fold'] == fold_id

    X_tr = df_fold.loc[train_mask, FEATURE_COLS].values.astype(np.float32)
    y_tr = df_fold.loc[train_mask, TARGET_COL].values.astype(np.float32)
    X_te = df_fold.loc[test_mask,  FEATURE_COLS].values.astype(np.float32)
    y_te = df_fold.loc[test_mask,  TARGET_COL].values.astype(np.float32)

    n_tr_st = df_fold.loc[train_mask, 'station_id'].nunique()
    n_te_st = df_fold.loc[test_mask,  'station_id'].nunique()
    print(f'  Train: {X_tr.shape}  ({n_tr_st} stations)')
    print(f'  Test:  {X_te.shape}  ({n_te_st} stations)')
    print(f'  NaN check — X_tr: {np.isnan(X_tr).sum()}  X_te: {np.isnan(X_te).sum()}')

    del df_geo, df_fold; gc.collect()

    # ── 3. Train 11 quantile models (parallel across GPUs) ────────────────────
    print(f'  Training {len(QUANTILES)} quantiles on {N_PARALLEL_QUANTILES} '
          f'{"GPU(s)" if DEVICE == "cuda" else "CPU"}...')
    q_preds, fold_models, fold_best_iters = train_quantiles_multigpu(
        X_tr, y_tr, X_te, y_te,
        num_boost_round = NUM_BOOST_ROUND,
        early_stopping  = EARLY_STOPPING,
        base_params     = LGB_BASE,
        label           = f'fold{fold_id}',
    )

    # Enforce quantile monotonicity
    q_preds = np.sort(q_preds, axis=1)

    # ── 4. Metrics ────────────────────────────────────────────────────────────
    crps = ps.crps_ensemble(y_te, q_preds).mean()
    mae  = float(np.abs(y_te - q_preds[:, QUANTILES.index(0.50)]).mean())
    rmse = float(np.sqrt(((y_te - q_preds[:, QUANTILES.index(0.50)]) ** 2).mean()))
    print(f'  CRPS={crps:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}')

    # ── 5. Save models ────────────────────────────────────────────────────────
    joblib.dump({
        'models':           fold_models,
        'quantiles':        QUANTILES,
        'feature_cols':     FEATURE_COLS,
        'best_iterations':  fold_best_iters,
        'random_seed':      RANDOM_SEED,
    }, mdl_local)
    s3_upload(mdl_local, mdl_s3)

    # ── 6. Checkpoint CV results ──────────────────────────────────────────────
    cv_results.append({
        'fold':             fold_id,
        'crps':             float(crps),
        'mae':              float(mae),
        'rmse':             float(rmse),
        'n_test':           int(len(y_te)),
        'n_test_stations':  int(n_te_st),
    })
    with open(CV_JSON_LOCAL, 'w') as f:
        json.dump(cv_results, f, indent=2)
    s3_upload(CV_JSON_LOCAL, CV_JSON_S3)

    del X_tr, y_tr, X_te, y_te, fold_models, q_preds; gc.collect()
    print(f'Fold {fold_id} complete.')



FOLD 0 / 4
  Features: computing (n_jobs=48)...
  [fold0] done — 17,406,541 rows → fold0_features.parquet                    
  ↑ s3://thesis-data-ismaktam/lgbm/kfold/fold0_features.parquet
  Train: (13885714, 32)  (1572 stations)
  Test:  (3520827, 32)  (394 stations)
  NaN check — X_tr: 0  X_te: 0
  Training 11 quantiles on 3 GPU(s)...
GPU1 α=0.10 done (best_iter=225)
GPU0 α=0.05 done (best_iter=319)
GPU2 α=0.20 done (best_iter=513)
GPU0 α=0.30 done (best_iter=404)
GPU1 α=0.40 done (best_iter=803)
GPU2 α=0.50 done (best_iter=568)
GPU0 α=0.60 done (best_iter=872)
GPU2 α=0.80 done (best_iter=654)
GPU1 α=0.70 done (best_iter=995)
GPU1 α=0.95 done (best_iter=156)
GPU0 α=0.90 done (best_iter=568)
  CRPS=0.8039  MAE=1.0663  RMSE=2.1641
  ↑ s3://thesis-data-ismaktam/lgbm/kfold/fold0_models.joblib
  ↑ s3://thesis-data-ismaktam/lgbm/kfold/cv_results.json
Fold 0 complete.

FOLD 1 / 4
  Features: computing (n_jobs=48)...
  [fold1] done — 17,407,073 rows → fold1_features.parquet                

## 7. CV Results Summary

In [13]:
with open(CV_JSON_LOCAL) as f:
    cv_results = json.load(f)

df_cv = pd.DataFrame(cv_results).set_index('fold')
summary = df_cv[['crps', 'mae', 'rmse']].agg(['mean', 'std'])

print('=== 5-Fold CV Results ===')
print(df_cv[['crps', 'mae', 'rmse', 'n_test', 'n_test_stations']].round(4).to_string())
print()
print('=== Summary (mean ± std) ===')
for col in ['crps', 'mae', 'rmse']:
    m, s = summary.loc['mean', col], summary.loc['std', col]
    print(f'  {col:<8} {m:.4f} ± {s:.4f}')

print()
print(f'Baseline (Tweedie):  CRPS=0.7895  MAE=0.790')
crps_mean = summary.loc['mean', 'crps']
print(f'Improvement CRPS:    {(0.7895 - crps_mean) / 0.7895 * 100:.1f}%')

=== 5-Fold CV Results ===
        crps     mae    rmse   n_test  n_test_stations
fold                                                  
0     0.8039  1.0663  2.1641  3520827              394
1     0.7962  1.0556  2.1409  3466145              393
2     0.7871  1.0400  2.1097  3495450              393
3     0.7742  1.0228  2.0928  3471152              393
4     0.7769  1.0239  2.1095  3452880              393

=== Summary (mean ± std) ===
  crps     0.7876 ± 0.0126
  mae      1.0417 ± 0.0192
  rmse     2.1234 ± 0.0286

Baseline (Tweedie):  CRPS=0.7895  MAE=0.790
Improvement CRPS:    0.2%


## 8. Final Model — All Data

Trains 11 quantile models on ALL wet station records (no held-out fold).  
All stations are in the neighbour pool — no leakage concern.

In [14]:
print('=== FINAL MODEL ===')

final_feat_local = FINAL_DIR / 'features.parquet'
final_feat_s3    = f'{S3_ROOT}/final/features.parquet'

if final_feat_local.exists() and not FORCE_RECOMPUTE:
    print('Final features: local cache')
elif not FORCE_RECOMPUTE and s3_download(final_feat_s3, final_feat_local):
    print('Final features: downloaded from S3')
else:
    print(f'Final features: computing (n_jobs={N_JOBS})...')
    all_sids = set(station_meta['station_id'])
    write_features(groups, all_sids, final_feat_local, 'final')
    s3_upload(final_feat_local, final_feat_s3)

# Build all-data array
df_geo_all = pd.read_parquet(final_feat_local)
df_final = (
    df_wet[['station_id', 'date', 'precip_mm', 'x_proj', 'y_proj', 'elevation_m']]
    .merge(df_geo_all, on=['station_id', 'date'], how='inner')
    .merge(soil_static[available_soil].reset_index(), on='station_id', how='left')
)
for v in available_soil:
    df_final[v] = df_final[v].fillna(df_final[v].median())

X_all = df_final[FEATURE_COLS].values.astype(np.float32)
y_all = df_final[TARGET_COL].values.astype(np.float32)
print(f'X_all: {X_all.shape}  NaN: {np.isnan(X_all).sum()}')

del df_geo_all, df_final; gc.collect()

=== FINAL MODEL ===
Final features: computing (n_jobs=48)...
  [final] done — 17,407,155 rows → features.parquet                    
  ↑ s3://thesis-data-ismaktam/lgbm/final/features.parquet
X_all: (17407155, 32)  NaN: 0


2936

In [15]:
final_mdl_local = FINAL_DIR / 'models.joblib'
final_mdl_s3    = f'{S3_ROOT}/final/models.joblib'

print(f'Training {len(QUANTILES)} quantile models on all data '
      f'({N_PARALLEL_QUANTILES} {"GPU(s)" if DEVICE == "cuda" else "CPU"}, '
      f'no early stopping)...')

# No validation set → no early stopping, full NUM_BOOST_ROUND used
_, final_models, final_n_trees = train_quantiles_multigpu(
    X_all, y_all,
    X_va=None, y_va=None,
    num_boost_round = NUM_BOOST_ROUND,
    early_stopping  = 0,
    base_params     = LGB_BASE,
    label           = 'final',
)

for alpha in QUANTILES:
    print(f'  α={alpha:.2f}  trees={final_n_trees[alpha]}')

joblib.dump({
    'models':       final_models,
    'quantiles':    QUANTILES,
    'feature_cols': FEATURE_COLS,
    'random_seed':  RANDOM_SEED,
}, final_mdl_local)
s3_upload(final_mdl_local, final_mdl_s3)

del X_all, y_all, final_models; gc.collect()
print('Final model saved.')


Training 11 quantile models on all data (3 GPU(s), no early stopping)...
GPU2 α=0.20 done (trees=1000)
GPU1 α=0.10 done (trees=1000)
GPU0 α=0.05 done (trees=1000)
GPU2 α=0.50 done (trees=1000)
GPU1 α=0.40 done (trees=1000)
GPU0 α=0.30 done (trees=1000)
GPU2 α=0.80 done (trees=1000)
GPU0 α=0.60 done (trees=1000)
GPU1 α=0.70 done (trees=1000)
GPU1 α=0.95 done (trees=1000)
GPU0 α=0.90 done (trees=1000)
  α=0.05  trees=1000
  α=0.10  trees=1000
  α=0.20  trees=1000
  α=0.30  trees=1000
  α=0.40  trees=1000
  α=0.50  trees=1000
  α=0.60  trees=1000
  α=0.70  trees=1000
  α=0.80  trees=1000
  α=0.90  trees=1000
  α=0.95  trees=1000
  ↑ s3://thesis-data-ismaktam/lgbm/final/models.joblib
Final model saved.


## 9. Done — S3 Manifest

In [16]:
print('=== ALL DONE ===')
print()
print(f'S3 outputs (s3://{S3_BUCKET}/{S3_ROOT}/)')
manifest = [
    'fold_assignment.parquet',
] + [
    f'kfold/fold{i}_{t}'
    for i in range(K_FOLDS)
    for t in ['features.parquet', 'models.joblib']
] + [
    'kfold/cv_results.json',
    'final/features.parquet',
    'final/models.joblib',
]

for key in manifest:
    full_key = f'{S3_ROOT}/{key}'
    exists   = s3_exists(full_key)
    status   = '✓' if exists else '✗ MISSING'
    print(f'  {status}  s3://{S3_BUCKET}/{full_key}')

print()
print('To evaluate: download from S3 and run lgbm_eval.ipynb')
print('The eval notebook loads fold_assignment.parquet from S3')
print('to reproduce the exact same folds as training.')

=== ALL DONE ===

S3 outputs (s3://thesis-data-ismaktam/lgbm/)
  ✓  s3://thesis-data-ismaktam/lgbm/fold_assignment.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold0_features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold0_models.joblib
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold1_features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold1_models.joblib
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold2_features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold2_models.joblib
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold3_features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold3_models.joblib
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold4_features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/fold4_models.joblib
  ✓  s3://thesis-data-ismaktam/lgbm/kfold/cv_results.json
  ✓  s3://thesis-data-ismaktam/lgbm/final/features.parquet
  ✓  s3://thesis-data-ismaktam/lgbm/final/models.joblib

To evaluate: download from S3 and run lgbm_eval.ipynb
The eval noteb